In [ ]:
import ee
import geemap.core as geemap
from datetime import datetime

ee.Authenticate()
ee.Initialize(project='sat-model-1')
print(ee.String('Hello from the Earth Engine servers!').getInfo())

In [ ]:
import pandas as pd

In [ ]:
df = pd.read_csv('model.csv', sep=';')

start_date = '2025-02-02'
end_date = '2025-02-04'

bathymetry = ee.ImageCollection("COPERNICUS/MARINE/WAV/ANFC_0_083DEG_STATIC") \
        .first() \
        .select('deptho')
        
oisst = ee.ImageCollection("NOAA/CDR/OISST/V2_1") \
    .filterDate('2023-07-15', '2023-07-16') \
    .mean() \
    .select('sst')

In [ ]:
def get_sst_at_point(image, lon, lat):
    point = ee.Geometry.Point([lon, lat])
    value = image.sample(point, 5000).first().get('sst')
    return value.getInfo() * 0.01  # масштабный коэффициент

df['sst'] = [get_sst_at_point(oisst, lon, lat) for lon, lat in zip(df['lon'], df['lat'])]

In [ ]:
def get_depth_at_point(image, lon, lat):
    point = ee.Geometry.Point([lon, lat])
    value = image.sample(point, 5000).first().get('deptho')
    return value.getInfo()  # масштабный коэффициент

df['depth'] = [get_depth_at_point(bathymetry, lon, lat) for lon, lat in zip(df['lon'], df['lat'])]

In [ ]:
df.head(10)

In [ ]:
df.to_csv('result.csv')

In [ ]:

m = geemap.Map()

vis_params = {
        'min': 0,
        'max': 6000,
        'palette': ['lightblue', 'blue', 'darkblue', 'black']
    }
    
m.add_layer(bathymetry, vis_params, 'Глубина (Copernicus)')

m.setCenter(150, 56, 2) 

In [ ]:
m